In [2]:
from __future__ import annotations
 
import os
import json
import time
import math
import random
import hashlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # safe for headless / notebook export
import matplotlib.pyplot as plt
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
 
import torchvision
from torchvision import transforms
from torchvision.models import (
    resnet50, ResNet50_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
)
 
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = False  # we audited: 0 corrupted images, keep it strict
 
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    classification_report,
)
 
 
# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------
@dataclass
class Config:
    # --- Paths -------------------------------------------------------------
    project_root: Path = Path(r"D:\nti_project")
    manifest_dir: Path = Path(r"D:\nti_project\data_processed\manifests")
    ckpt_dir: Path = Path(r"D:\nti_project\models\classification")
    artifact_dir: Path = Path(r"D:\nti_project\audit_results\classification")
 
    # --- Manifest schema ---------------------------------------------------
    # Candidate column names; the resolver in CELL 2 picks the first that exists.
    path_col_candidates: Sequence[str] = (
        "image_path", "filepath", "path", "abs_path", "image", "file_path",
    )
    label_col_candidates: Sequence[str] = (
        "label", "class", "class_name", "category", "folder", "source_folder",
        "subfolder", "target",
    )
    task_col_candidates: Sequence[str] = ("task", "subtask", "dataset", "split_task")
 
    # Raw folder name -> binary label. Lower-cased, whitespace-normalised.
    label_map: dict = field(default_factory=lambda: {
        "positive": 1,
        "crack": 1,
        "damage": 1,
        "negative": 0,
        "no crack": 0,
        "no_crack": 0,
        "nocrack": 0,
        "background": 0,
    })
    class_names: Sequence[str] = ("No-Damage", "Damage")
 
    # --- Model -------------------------------------------------------------
    arch: str = "resnet50"          # "resnet50" | "efficientnet_b0"
    img_size: int = 224
    dropout: float = 0.3
 
    # --- Training ----------------------------------------------------------
    batch_size: int = 32            # 224px + AMP + ResNet50 ≈ 4.5 GB on 6 GB card
    accum_steps: int = 1            # raise to 2 if you hit OOM and halve batch_size
    num_workers: int = 4            # Windows: keep <= 4, persistent_workers avoids respawn cost
    epochs_head: int = 3            # stage 1: frozen backbone, train classifier head
    epochs_full: int = 25           # stage 2: full fine-tune
    lr_head: float = 1e-3
    lr_backbone: float = 1e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05
    warmup_epochs: int = 1
    early_stopping_patience: int = 6
    monitor_metric: str = "f1"      # "f1" | "auc" | "acc" — selection metric on VAL
    grad_clip: float = 1.0
 
    # --- Imbalance handling ------------------------------------------------
    # "sampler" = WeightedRandomSampler, "weights" = class-weighted loss,
    # "focal" = focal loss, "none" = plain CE
    imbalance_strategy: str = "sampler"
    focal_gamma: float = 2.0
 
    # --- Misc --------------------------------------------------------------
    seed: int = 42
    amp: bool = True
    channels_last: bool = True
    limit_to_classification_task: bool = True  # drop detection/seg rows if a task col exists
 
 
CFG = Config()
CFG.ckpt_dir.mkdir(parents=True, exist_ok=True)
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
 
def set_seed(seed: int) -> None:
    """Full reproducibility across python / numpy / torch / cudnn."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # deterministic=False keeps cudnn autotuner on (much faster); flip if you need
    # bit-exact reruns.
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
 
 
set_seed(CFG.seed)
 
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"device      : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"gpu         : {torch.cuda.get_device_name(0)}")
    print(f"vram        : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
 

torch       : 2.5.1+cu121
torchvision : 0.20.1+cu121
device      : cuda
gpu         : NVIDIA GeForce RTX 3050 6GB Laptop GPU
vram        : 6.0 GB


In [3]:
def _resolve_column(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
    """Return the first candidate column present in df (case-insensitive)."""
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None
 
 
def load_manifest(split: str) -> pd.DataFrame:
    fp = CFG.manifest_dir / f"{split}_manifest.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Manifest not found: {fp}")
    df = pd.read_csv(fp)
    df["__split__"] = split
    return df
 
 
raw_splits = {s: load_manifest(s) for s in ("train", "val", "test")}
 
print("=" * 78)
print("MANIFEST SCHEMA INSPECTION")
print("=" * 78)
for s, df in raw_splits.items():
    print(f"\n[{s}] rows={len(df):,}  columns={list(df.columns)}")
    print(df.head(3).to_string())
 
PATH_COL = _resolve_column(raw_splits["train"], CFG.path_col_candidates)
LABEL_COL = _resolve_column(raw_splits["train"], CFG.label_col_candidates)
TASK_COL = _resolve_column(raw_splits["train"], CFG.task_col_candidates)
 
print("\n" + "-" * 78)
print(f"Resolved path column  : {PATH_COL}")
print(f"Resolved label column : {LABEL_COL}")
print(f"Resolved task column  : {TASK_COL}")
print("-" * 78)
 
assert PATH_COL is not None, (
    "Could not resolve the image-path column. Add its name to "
    "CFG.path_col_candidates and re-run this cell."
)
assert LABEL_COL is not None, (
    "Could not resolve the label/folder column. Add its name to "
    "CFG.label_col_candidates and re-run this cell."
)

MANIFEST SCHEMA INSPECTION

[train] rows=44,301  columns=['image_path', 'filename', 'width', 'height', 'label', 'target', 'mask_path', 'group_id', 'split', '__split__']
                                                                                image_path          filename  width  height      label  target                                                         mask_path          group_id  split __split__
0  D:\nti_project\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\grace_hopper.jpg  grace_hopper.jpg    512     600  No-Damage       0  D:\nti_project\data_processed\masks_binary\grace_hopper_mask.png  grace_hopper.jpg  train     train
1                 D:\nti_project\.venv\Lib\site-packages\sklearn\datasets\images\china.jpg         china.jpg    640     427  No-Damage       0         D:\nti_project\data_processed\masks_binary\china_mask.png         china.jpg  train     train
2                                                  D:\nti_project\data\Background\0001.jpg         

In [4]:
def normalize_label(value: str) -> str:
    return str(value).strip().lower().replace("\\", "/").split("/")[-1]
 
 
def build_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
 
    # 1) Restrict to classification-relevant rows if the manifest is multi-task.
    if TASK_COL is not None and CFG.limit_to_classification_task:
        mask = out[TASK_COL].astype(str).str.lower().str.contains("class", na=False)
        if mask.any():
            out = out[mask]
 
    # 2) Map raw folder names to binary labels.
    out["_raw_label"] = out[LABEL_COL].map(normalize_label)
    out["label"] = out["_raw_label"].map(CFG.label_map)
 
    unmapped = out.loc[out["label"].isna(), "_raw_label"].unique().tolist()
    if unmapped:
        print(f"  [warn] dropping {len(out[out['label'].isna()]):,} rows with "
              f"unmapped labels: {unmapped}")
        print("         -> add them to CFG.label_map if they belong in this task.")
        out = out.dropna(subset=["label"])
 
    out["label"] = out["label"].astype(int)
 
    # 3) Resolve paths to absolute, verify existence.
    def _abs(p: str) -> str:
        p = str(p)
        return p if os.path.isabs(p) else str(CFG.project_root / p)
 
    out["abs_path"] = out[PATH_COL].map(_abs)
    exists = out["abs_path"].map(os.path.exists)
    missing = int((~exists).sum())
    if missing:
        print(f"  [warn] {missing:,} files listed in the manifest are missing on disk "
              f"— dropped.")
        out = out[exists]
 
    return out[["abs_path", "label", "_raw_label", "__split__"]].reset_index(drop=True)
 
 
frames = {}
for split, df in raw_splits.items():
    print(f"\nBuilding [{split}] …")
    frames[split] = build_frame(df)
 
train_df, val_df, test_df = frames["train"], frames["val"], frames["test"]
 
# -----------------------------------------------------------------------------
# LEAKAGE GUARD — a hard assertion, not a comment.
# The MD5 group split was done in Notebook 03; this re-verifies it at the file
# level so a silent regression upstream cannot poison Phase 3 results.
# -----------------------------------------------------------------------------
def _basenames(df: pd.DataFrame) -> set:
    return set(Path(p).name.lower() for p in df["abs_path"])
 
 
tr_b, va_b, te_b = _basenames(train_df), _basenames(val_df), _basenames(test_df)
overlaps = {
    "train∩val": tr_b & va_b,
    "train∩test": tr_b & te_b,
    "val∩test": va_b & te_b,
}
for k, v in overlaps.items():
    print(f"  filename overlap {k}: {len(v)}")
assert not any(overlaps.values()), (
    "LEAKAGE DETECTED — identical filenames appear across splits. "
    "Re-run Notebook 03's MD5 group split before training."
)
 
print("\n" + "=" * 78)
print("CLASSIFICATION DATASET SUMMARY")
print("=" * 78)
for split, df in (("train", train_df), ("val", val_df), ("test", test_df)):
    counts = df["label"].value_counts().sort_index()
    n0, n1 = int(counts.get(0, 0)), int(counts.get(1, 0))
    total = n0 + n1
    print(f"{split:5s} | total {total:7,} | No-Damage {n0:7,} ({n0/max(total,1):5.1%}) "
          f"| Damage {n1:7,} ({n1/max(total,1):5.1%})")
 
print("\nSource-folder breakdown (train):")
print(train_df.groupby(["_raw_label", "label"]).size().to_string())
 


Building [train] …
  [warn] dropping 18,256 rows with unmapped labels: ['no-damage']
         -> add them to CFG.label_map if they belong in this task.

Building [val] …
  [warn] dropping 2,267 rows with unmapped labels: ['no-damage']
         -> add them to CFG.label_map if they belong in this task.

Building [test] …
  [warn] dropping 2,290 rows with unmapped labels: ['no-damage']
         -> add them to CFG.label_map if they belong in this task.
  filename overlap train∩val: 0
  filename overlap train∩test: 0
  filename overlap val∩test: 0

CLASSIFICATION DATASET SUMMARY
train | total  26,045 | No-Damage       0 ( 0.0%) | Damage  26,045 (100.0%)
val   | total   3,269 | No-Damage       0 ( 0.0%) | Damage   3,269 (100.0%)
test  | total   3,262 | No-Damage       0 ( 0.0%) | Damage   3,262 (100.0%)

Source-folder breakdown (train):
_raw_label  label
damage      1        26045


In [5]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
 
# Augmentations kept physically plausible for concrete imagery:
# flips/rotations are valid (cracks have no canonical orientation), colour jitter
# simulates lighting/weathering, but no heavy distortion that would alter the
# apparent crack geometry the severity engine will later depend on.
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(CFG.img_size, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.25),
    transforms.RandomApply([transforms.RandomRotation(20)], p=0.4),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.02),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.2))], p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.10), value="random"),
])
 
eval_tf = transforms.Compose([
    transforms.Resize(int(CFG.img_size * 1.14)),
    transforms.CenterCrop(CFG.img_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
 
 
class ConcreteDamageDataset(Dataset):
    """Reads (path, binary label) pairs straight from a Phase-3 manifest frame."""
 
    def __init__(self, df: pd.DataFrame, transform, return_path: bool = False):
        self.paths = df["abs_path"].tolist()
        self.labels = df["label"].astype(np.int64).tolist()
        self.transform = transform
        self.return_path = return_path
 
    def __len__(self) -> int:
        return len(self.paths)
 
    def __getitem__(self, idx: int):
        path = self.paths[idx]
        with Image.open(path) as im:
            img = im.convert("RGB")  # grayscale + RGBA sources normalised here
        x = self.transform(img)
        y = self.labels[idx]
        if self.return_path:
            return x, y, path
        return x, y
 
 
def make_loaders() -> tuple[DataLoader, DataLoader]:
    train_ds = ConcreteDamageDataset(train_df, train_tf)
    val_ds = ConcreteDamageDataset(val_df, eval_tf)
 
    sampler, shuffle = None, True
    if CFG.imbalance_strategy == "sampler":
        counts = np.bincount(train_df["label"].values, minlength=2).astype(float)
        per_class_w = 1.0 / np.maximum(counts, 1.0)
        sample_w = per_class_w[train_df["label"].values]
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(sample_w, dtype=torch.double),
            num_samples=len(sample_w),
            replacement=True,
        )
        shuffle = False
        print(f"WeightedRandomSampler active — class counts {counts.tolist()}, "
              f"weights {per_class_w.round(6).tolist()}")
 
    common = dict(
        batch_size=CFG.batch_size,
        num_workers=CFG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=CFG.num_workers > 0,
        prefetch_factor=4 if CFG.num_workers > 0 else None,
    )
    train_loader = DataLoader(train_ds, sampler=sampler, shuffle=shuffle,
                              drop_last=True, **common)
    val_loader = DataLoader(val_ds, shuffle=False, drop_last=False, **common)
    return train_loader, val_loader
 
 
train_loader, val_loader = make_loaders()
print(f"train batches: {len(train_loader):,} | val batches: {len(val_loader):,}")
 

WeightedRandomSampler active — class counts [0.0, 26045.0], weights [1.0, 3.8e-05]
train batches: 813 | val batches: 103


In [6]:
def build_model(arch: str, num_classes: int = 2, dropout: float = 0.3) -> nn.Module:
    """Transfer-learning backbone with a fresh classifier head."""
    if arch == "resnet50":
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        in_f = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
        head_names = ("fc",)
    elif arch == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
        head_names = ("classifier",)
    else:
        raise ValueError(f"Unsupported arch: {arch}")
    return model, head_names
 
 
def set_backbone_trainable(model: nn.Module, head_names: Sequence[str], flag: bool) -> None:
    for name, p in model.named_parameters():
        is_head = any(name.startswith(h) for h in head_names)
        p.requires_grad = True if is_head else flag
 
 
class FocalLoss(nn.Module):
    """Multi-class focal loss — useful if you later extend to defect sub-types
    where the minority class ratio matches the detection split (962 / 39,420)."""
 
    def __init__(self, gamma: float = 2.0, weight: Optional[torch.Tensor] = None,
                 label_smoothing: float = 0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing
 
    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, target, weight=self.weight,
                             label_smoothing=self.label_smoothing, reduction="none")
        pt = torch.exp(-ce)
        return ((1.0 - pt) ** self.gamma * ce).mean()
 
 
def build_criterion() -> nn.Module:
    class_w = None
    if CFG.imbalance_strategy in ("weights", "focal"):
        counts = np.bincount(train_df["label"].values, minlength=2).astype(float)
        w = counts.sum() / (2.0 * np.maximum(counts, 1.0))
        class_w = torch.tensor(w, dtype=torch.float32, device=DEVICE)
        print(f"class weights: {w.round(4).tolist()}")
 
    if CFG.imbalance_strategy == "focal":
        return FocalLoss(CFG.focal_gamma, class_w, CFG.label_smoothing)
    return nn.CrossEntropyLoss(weight=class_w, label_smoothing=CFG.label_smoothing)
 
 
model, HEAD_NAMES = build_model(CFG.arch, 2, CFG.dropout)
model = model.to(DEVICE)
if CFG.channels_last and DEVICE.type == "cuda":
    model = model.to(memory_format=torch.channels_last)
 
criterion = build_criterion()
scaler = torch.amp.GradScaler("cuda", enabled=(CFG.amp and DEVICE.type == "cuda"))
 
n_total = sum(p.numel() for p in model.parameters())
print(f"{CFG.arch}: {n_total/1e6:.2f} M parameters")
 

resnet50: 23.51 M parameters


In [7]:
from pathlib import Path

base_dir = Path(r"D:\nti_project\data_processed")
print(f"Base folder exists: {base_dir.exists()}")

if base_dir.exists():
    print("\nCSV files found:")
    csv_files = list(base_dir.rglob("*.csv"))
    if csv_files:
        for f in csv_files:
            print(f" - {f}")
    else:
        print(" No CSV manifest files found!")

Base folder exists: True

CSV files found:
 - D:\nti_project\data_processed\manifests\full_manifest.csv
 - D:\nti_project\data_processed\manifests\test_manifest.csv
 - D:\nti_project\data_processed\manifests\train_manifest.csv
 - D:\nti_project\data_processed\manifests\val_manifest.csv


In [8]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module,
             threshold: float = 0.5) -> dict:
    model.eval()
    losses, probs, targets = [], [], []
 
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        if CFG.channels_last and DEVICE.type == "cuda":
            x = x.to(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
 
        with torch.amp.autocast("cuda", enabled=(CFG.amp and DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)
 
        losses.append(loss.item() * x.size(0))
        probs.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
        targets.append(y.cpu().numpy())
 
    p = np.concatenate(probs)
    t = np.concatenate(targets)
    pred = (p >= threshold).astype(int)
 
    prec, rec, f1, _ = precision_recall_fscore_support(
        t, pred, average="binary", zero_division=0)
    metrics = {
        "loss": float(np.sum(losses) / max(len(t), 1)),
        "acc": float(accuracy_score(t, pred)),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "auc": float(roc_auc_score(t, p)) if len(np.unique(t)) > 1 else float("nan"),
        "ap": float(average_precision_score(t, p)) if len(np.unique(t)) > 1 else float("nan"),
        "threshold": threshold,
    }
    return metrics, p, t
 
 
def train_one_epoch(model, loader, criterion, optimizer, scheduler, epoch: int) -> dict:
    model.train()
    running, seen, correct = 0.0, 0, 0
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)
 
    for step, (x, y) in enumerate(loader):
        x = x.to(DEVICE, non_blocking=True)
        if CFG.channels_last and DEVICE.type == "cuda":
            x = x.to(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
 
        with torch.amp.autocast("cuda", enabled=(CFG.amp and DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y) / CFG.accum_steps
 
        scaler.scale(loss).backward()
 
        if (step + 1) % CFG.accum_steps == 0:
            if CFG.grad_clip:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None:
                scheduler.step()
 
        bs = x.size(0)
        running += loss.item() * CFG.accum_steps * bs
        seen += bs
        correct += (logits.argmax(1) == y).sum().item()
 
        if step % 50 == 0:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"  ep{epoch:02d} step {step:5d}/{len(loader)} "
                  f"loss {running/max(seen,1):.4f} acc {correct/max(seen,1):.4f} "
                  f"lr {lr_now:.2e}", flush=True)
 
    return {
        "loss": running / max(seen, 1),
        "acc": correct / max(seen, 1),
        "secs": time.time() - t0,
    }
 

In [9]:
# %% [markdown]
# # Phase 1: Notebook 03 — Fast Parallel Preprocessing & Group Split

# %%
from __future__ import annotations

import os
import hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

# 1. CONFIGURATION
@dataclass
class Config:
    project_root: Path = Path(r"D:\nti_project")
    processed_dir: Path = Path(r"D:\nti_project\data_processed")
    manifest_dir: Path = Path(r"D:\nti_project\data_processed\manifests")
    masks_out_dir: Path = Path(r"D:\nti_project\data_processed\masks_binary")
    train_ratio: float = 0.80
    val_ratio: float = 0.10
    test_ratio: float = 0.10
    seed: int = 42

CFG = Config()
CFG.manifest_dir.mkdir(parents=True, exist_ok=True)
CFG.masks_out_dir.mkdir(parents=True, exist_ok=True)

# 2. SCAN IMAGES
image_extensions = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
image_files = []
for ext in image_extensions:
    image_files.extend(list(CFG.project_root.rglob(ext)))

image_files = [
    f for f in image_files 
    if "data_processed" not in f.parts and "audit_results" not in f.parts
]
print(f"Found {len(image_files):,} images. Processing in parallel...")

# 3. FAST PARALLEL WORKER
def process_image(img_path: Path) -> Dict[str, Any] | None:
    try:
        path_str = str(img_path).lower()
        label_str = "Damage" if any(k in path_str for k in ["positive", "crack", "damage", "spalling"]) else "No-Damage"
        label_val = 1 if label_str == "Damage" else 0
        
        mask_target_path = CFG.masks_out_dir / f"{img_path.stem}_mask.png"
        
        # Fast-track: skip cv2 write if mask already created during the previous run
        if not mask_target_path.exists():
            with Image.open(img_path) as img:
                w, h = img.size
            mask = np.full((h, w), 255, dtype=np.uint8) if label_val == 1 else np.zeros((h, w), dtype=np.uint8)
            cv2.imwrite(str(mask_target_path), mask)
        else:
            # Quick dimension fetch without heavy decoding
            with Image.open(img_path) as img:
                w, h = img.size

        return {
            "image_path": str(img_path),
            "filename": img_path.name,
            "width": w,
            "height": h,
            "label": label_str,
            "target": label_val,
            "mask_path": str(mask_target_path),
            "group_id": img_path.name.lower()  # Filename-based grouping prevents cross-split overlap
        }
    except Exception:
        return None

# Execute with 16 Workers (Multi-threading)
with ThreadPoolExecutor(max_workers=16) as executor:
    results = list(executor.map(process_image, image_files))

records = [r for r in results if r is not None]
df_all = pd.DataFrame(records)
print(f"Successfully processed {len(df_all):,} items.")

# 4. ZERO-LEAKAGE GROUP SPLIT
np.random.seed(CFG.seed)
gss_test = GroupShuffleSplit(n_splits=1, test_size=CFG.test_ratio, random_state=CFG.seed)
train_val_idx, test_idx = next(gss_test.split(df_all, groups=df_all["group_id"]))

df_train_val = df_all.iloc[train_val_idx].copy()
df_test = df_all.iloc[test_idx].copy()

val_rel = CFG.val_ratio / (CFG.train_ratio + CFG.val_ratio)
gss_val = GroupShuffleSplit(n_splits=1, test_size=val_rel, random_state=CFG.seed)
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val["group_id"]))

df_train = df_train_val.iloc[train_idx].copy()
df_val = df_train_val.iloc[val_idx].copy()

df_train["split"], df_val["split"], df_test["split"] = "train", "val", "test"

# 5. EXPORT MANIFESTS
df_train.to_csv(CFG.manifest_dir / "train_manifest.csv", index=False)
df_val.to_csv(CFG.manifest_dir / "val_manifest.csv", index=False)
df_test.to_csv(CFG.manifest_dir / "test_manifest.csv", index=False)
pd.concat([df_train, df_val, df_test], ignore_index=True).to_csv(CFG.manifest_dir / "full_manifest.csv", index=False)

print("\nSUCCESS: All manifest CSVs generated!")

Found 55,389 images. Processing in parallel...
Successfully processed 55,389 items.

SUCCESS: All manifest CSVs generated!


In [10]:
import torch
from pathlib import Path

# 1. إجبار إعادة إنشاء وتحديث كائن CFG بكافة الخصائص
class Config:
    project_root: Path = Path(r"D:\nti_project")
    manifest_dir: Path = Path(r"D:\nti_project\data_processed\manifests")
    ckpt_dir: Path = Path(r"D:\nti_project\models\classification")
    artifact_dir: Path = Path(r"D:\nti_project\audit_results\classification")
    batch_size: int = 32
    num_workers: int = 0
    amp: bool = True
    channels_last: bool = True
    lr_head: float = 1e-3
    lr_backbone: float = 1e-4
    weight_decay: float = 1e-4
    accum_steps: int = 1
    epochs_full: int = 25
    warmup_epochs: int = 1
    early_stopping_patience: int = 6
    monitor_metric: str = "f1"
    arch: str = "resnet50"
    img_size: int = 224
    class_names: tuple = ("No-Damage", "Damage")
    imbalance_strategy: str = "sampler"
    grad_clip: float = 1.0

# إعادة التكويين دائماً لتحديث الخصائص في الذاكرة
CFG = Config()

# 2. التأكد من وجود DEVICE
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. تهيئة AMP GradScaler
scaler = torch.amp.GradScaler("cuda", enabled=(CFG.amp and DEVICE.type == "cuda"))

# 4. تحديث الـ Loaders لمنع تعليق Windows
if 'train_dataset' in globals() and 'val_dataset' in globals():
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=CFG.batch_size,
        sampler=train_sampler if (CFG.imbalance_strategy == "sampler" and 'train_sampler' in globals()) else None,
        shuffle=('train_sampler' not in globals() or train_sampler is None),
        num_workers=CFG.num_workers,
        pin_memory=True,
        drop_last=True,
    )

    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=True,
    )

print(f"✓ Environment Ready | Device: {DEVICE} | Scaler Initialized | num_workers={CFG.num_workers}")

✓ Environment Ready | Device: cuda | Scaler Initialized | num_workers=0


In [13]:
import os
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

# 1. Device & Config Setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Config:
    project_root: Path = Path(r"D:\nti_project")
    manifest_dir: Path = Path(r"D:\nti_project\data_processed\manifests")
    ckpt_dir: Path = Path(r"D:\nti_project\models\classification")
    artifact_dir: Path = Path(r"D:\nti_project\audit_results\classification")
    batch_size: int = 32
    img_size: int = 224
    num_workers: int = 0  # 0 to avoid Windows multiprocessing deadlock
    amp: bool = True
    lr_head: float = 1e-3
    lr_backbone: float = 1e-4
    weight_decay: float = 1e-4
    accum_steps: int = 1
    epochs_full: int = 25
    warmup_epochs: int = 1
    early_stopping_patience: int = 6
    monitor_metric: str = "f1"
    arch: str = "resnet50"
    class_names: tuple = ("No-Damage", "Damage")
    imbalance_strategy: str = "sampler"
    grad_clip: float = 1.0

CFG = Config()
CFG.ckpt_dir.mkdir(parents=True, exist_ok=True)
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

# 2. Transforms & Custom Dataset
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ConcreteDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.paths = self.df["image_path"].tolist()
        self.targets = self.df["target"].values.astype(np.int64)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.targets[idx], dtype=torch.long)

# Load CSV Manifests
df_train = pd.read_csv(CFG.manifest_dir / "train_manifest.csv")
df_val = pd.read_csv(CFG.manifest_dir / "val_manifest.csv")

train_dataset = ConcreteDataset(df_train, transform=train_tfms)
val_dataset = ConcreteDataset(df_val, transform=val_tfms)

# Weighted Sampler for Class Imbalance
class_counts = np.bincount(train_dataset.targets)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_dataset.targets]
train_sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    sampler=train_sampler,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

# 3. Model Architecture
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 2)
model = model.to(DEVICE)

HEAD_NAMES = ("fc", "classifier", "head")

def set_backbone_trainable(m: nn.Module, head_names: tuple, flag: bool = True):
    for name, param in m.named_parameters():
        if not any(name.startswith(h) for h in head_names):
            param.requires_grad = flag

set_backbone_trainable(model, HEAD_NAMES, flag=True)

criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda", enabled=(CFG.amp and DEVICE.type == "cuda"))

backbone_params = [p for n, p in model.named_parameters() if not any(n.startswith(h) for h in HEAD_NAMES)]
head_params = [p for n, p in model.named_parameters() if any(n.startswith(h) for h in HEAD_NAMES)]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": CFG.lr_backbone},
        {"params": head_params, "lr": CFG.lr_head * 0.1},
    ],
    weight_decay=CFG.weight_decay,
)

steps_per_epoch = math.ceil(len(train_loader) / CFG.accum_steps)
total_steps = steps_per_epoch * CFG.epochs_full
warmup_steps = steps_per_epoch * CFG.warmup_epochs

def lr_lambda(step: int) -> float:
    if step < warmup_steps:
        return (step + 1) / max(warmup_steps, 1)
    prog = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1.0 + math.cos(math.pi * min(prog, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# 4. Train & Eval Loop Definitions
@torch.no_grad()
def evaluate(model, loader, criterion, threshold=0.5):
    model.eval()
    losses, probs, targets = [], [], []
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(CFG.amp and DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        losses.append(loss.item() * x.size(0))
        probs.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
        targets.append(y.cpu().numpy())
    p = np.concatenate(probs)
    t = np.concatenate(targets)
    pred = (p >= threshold).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(t, pred, average="binary", zero_division=0)
    metrics = {
        "loss": float(np.sum(losses) / max(len(t), 1)),
        "acc": float(accuracy_score(t, pred)),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "auc": float(roc_auc_score(t, p)) if len(np.unique(t)) > 1 else float("nan"),
    }
    return metrics, p, t

def train_one_epoch(model, loader, criterion, optimizer, scheduler, epoch):
    model.train()
    running, seen, correct = 0.0, 0, 0
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)
    for step, (x, y) in enumerate(loader):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(CFG.amp and DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y) / CFG.accum_steps
        scaler.scale(loss).backward()
        if (step + 1) % CFG.accum_steps == 0:
            if CFG.grad_clip:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None:
                scheduler.step()
        bs = x.size(0)
        running += loss.item() * CFG.accum_steps * bs
        seen += bs
        correct += (logits.argmax(1) == y).sum().item()
        if step % 50 == 0:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"  ep{epoch:02d} step {step:5d}/{len(loader)} loss {running/max(seen,1):.4f} acc {correct/max(seen,1):.4f} lr {lr_now:.2e}", flush=True)
    return {"loss": running / max(seen, 1), "acc": correct / max(seen, 1), "secs": time.time() - t0}

# 5. Execute Stage 2 Training
BEST_CKPT = CFG.ckpt_dir / "best_model.pth"
LAST_CKPT = CFG.ckpt_dir / "last_model.pth"

history, best_score, best_epoch, patience = [], -np.inf, -1, 0

print("=" * 78)
print(f"STAGE 2 — full fine-tune | Device: {DEVICE} | monitoring val/{CFG.monitor_metric}")
print("=" * 78)

for ep in range(1, CFG.epochs_full + 1):
    tr = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, ep)
    va, va_probs, va_targets = evaluate(model, val_loader, criterion)

    row = {
        "epoch": ep,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": tr["loss"],
        "train_acc": tr["acc"],
        **{f"val_{k}": v for k, v in va.items()},
    }
    history.append(row)

    print(
        f"[ep {ep:02d}/{CFG.epochs_full}] train loss {tr['loss']:.4f} acc {tr['acc']:.4f} "
        f"| val loss {va['loss']:.4f} acc {va['acc']:.4f} "
        f"P {va['precision']:.4f} R {va['recall']:.4f} F1 {va['f1']:.4f} "
        f"AUC {va['auc']:.4f} | {tr['secs']:.0f}s"
    )

    score = va[CFG.monitor_metric]
    if score > best_score:
        best_score, best_epoch, patience = score, ep, 0
        torch.save({
            "epoch": ep,
            "arch": CFG.arch,
            "img_size": CFG.img_size,
            "class_names": list(CFG.class_names),
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_metrics": va,
        }, BEST_CKPT)
        np.savez(CFG.artifact_dir / "val_predictions_best.npz", probs=va_probs, targets=va_targets)
        print(f"    ✓ new best val/{CFG.monitor_metric} = {score:.4f} → {BEST_CKPT.name}")
    else:
        patience += 1
        if patience >= CFG.early_stopping_patience:
            print(f"    ⏹ early stopping at epoch {ep} (best {CFG.monitor_metric}={best_score:.4f} @ ep{best_epoch})")
            break

torch.save({"epoch": ep, "state_dict": model.state_dict()}, LAST_CKPT)
hist_df = pd.DataFrame(history)
hist_df.to_csv(CFG.artifact_dir / "training_history.csv", index=False)
print(f"\nBest val/{CFG.monitor_metric} = {best_score:.4f} at epoch {best_epoch}")

STAGE 2 — full fine-tune | Device: cuda | monitoring val/f1
  ep01 step     0/1384 loss 0.6702 acc 0.5938 lr 1.45e-07
  ep01 step    50/1384 loss 0.6971 acc 0.5006 lr 3.76e-06
  ep01 step   100/1384 loss 0.6754 acc 0.5408 lr 7.37e-06
  ep01 step   150/1384 loss 0.6383 acc 0.6175 lr 1.10e-05
  ep01 step   200/1384 loss 0.5798 acc 0.6915 lr 1.46e-05
  ep01 step   250/1384 loss 0.5092 acc 0.7423 lr 1.82e-05
  ep01 step   300/1384 loss 0.4487 acc 0.7765 lr 2.18e-05
  ep01 step   350/1384 loss 0.3983 acc 0.8037 lr 2.54e-05
  ep01 step   400/1384 loss 0.3582 acc 0.8247 lr 2.90e-05
  ep01 step   450/1384 loss 0.3274 acc 0.8408 lr 3.27e-05
  ep01 step   500/1384 loss 0.3004 acc 0.8550 lr 3.63e-05
  ep01 step   550/1384 loss 0.2766 acc 0.8672 lr 3.99e-05
  ep01 step   600/1384 loss 0.2585 acc 0.8769 lr 4.35e-05
  ep01 step   650/1384 loss 0.2418 acc 0.8853 lr 4.71e-05
  ep01 step   700/1384 loss 0.2282 acc 0.8922 lr 5.07e-05
  ep01 step   750/1384 loss 0.2164 acc 0.8982 lr 5.43e-05
  ep01 step 

In [14]:
# Threshold is tuned on VALIDATION. The test set stays sealed until Notebook 07.
ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']}")
 
va_metrics, va_probs, va_targets = evaluate(model, val_loader, criterion)
 
# --- Sweep thresholds; pick the F1-optimal operating point -------------------
ths = np.linspace(0.05, 0.95, 91)
rows = []
for th in ths:
    pred = (va_probs >= th).astype(int)
    p, r, f, _ = precision_recall_fscore_support(va_targets, pred, average="binary",
                                                 zero_division=0)
    rows.append({"threshold": th, "precision": p, "recall": r, "f1": f})
sweep = pd.DataFrame(rows)
BEST_TH = float(sweep.loc[sweep["f1"].idxmax(), "threshold"])
 
# For an inspection system, a missed defect costs far more than a false alarm.
# This reports the highest-precision threshold that still holds recall >= 0.98,
# which is usually the right operating point to ship for the triage gate.
safe = sweep[sweep["recall"] >= 0.98]
SAFETY_TH = float(safe.loc[safe["precision"].idxmax(), "threshold"]) if len(safe) else BEST_TH
 
print(f"F1-optimal threshold        : {BEST_TH:.3f}")
print(f"High-recall (≥0.98) threshold: {SAFETY_TH:.3f}")
 
final_metrics, _, _ = evaluate(model, val_loader, criterion, threshold=BEST_TH)
print("\nVAL metrics @ tuned threshold:")
for k, v in final_metrics.items():
    print(f"  {k:10s}: {v:.4f}")
 
pred_at_best = (va_probs >= BEST_TH).astype(int)
cm = confusion_matrix(va_targets, pred_at_best)
print("\nClassification report (val @ tuned threshold):")
print(classification_report(va_targets, pred_at_best,
                            target_names=list(CFG.class_names), digits=4))
 
# --- Plots -------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
 
axes[0, 0].plot(hist_df["epoch"], hist_df["train_loss"], label="train")
axes[0, 0].plot(hist_df["epoch"], hist_df["val_loss"], label="val")
axes[0, 0].set_title("Loss"); axes[0, 0].set_xlabel("epoch"); axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
 
axes[0, 1].plot(hist_df["epoch"], hist_df["val_acc"], label="val acc")
axes[0, 1].plot(hist_df["epoch"], hist_df["val_f1"], label="val F1")
axes[0, 1].plot(hist_df["epoch"], hist_df["val_auc"], label="val AUC")
axes[0, 1].set_title("Validation metrics"); axes[0, 1].set_xlabel("epoch")
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)
 
fpr, tpr, _ = roc_curve(va_targets, va_probs)
axes[1, 0].plot(fpr, tpr, label=f"AUC = {va_metrics['auc']:.4f}")
axes[1, 0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1, 0].set_title("ROC"); axes[1, 0].set_xlabel("FPR"); axes[1, 0].set_ylabel("TPR")
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)
 
im = axes[1, 1].imshow(cm, cmap="Blues")
axes[1, 1].set_title(f"Confusion matrix (val @ th={BEST_TH:.2f})")
axes[1, 1].set_xticks([0, 1], CFG.class_names)
axes[1, 1].set_yticks([0, 1], CFG.class_names)
axes[1, 1].set_xlabel("predicted"); axes[1, 1].set_ylabel("true")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1, 1].text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=axes[1, 1], fraction=0.046)
 
plt.tight_layout()
fig.savefig(CFG.artifact_dir / "classification_diagnostics.png", dpi=150)
plt.close(fig)
print(f"\nSaved plots → {CFG.artifact_dir / 'classification_diagnostics.png'}")
 
# --- Persist the operating point so the inference engine reads it, not guesses -
with open(CFG.ckpt_dir / "classifier_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "arch": CFG.arch,
        "img_size": CFG.img_size,
        "class_names": list(CFG.class_names),
        "normalization": {"mean": list(IMAGENET_MEAN), "std": list(IMAGENET_STD)},
        "best_epoch": int(ckpt["epoch"]),
        "threshold_f1_optimal": BEST_TH,
        "threshold_high_recall": SAFETY_TH,
        "val_metrics_at_f1_threshold": final_metrics,
    }, f, indent=2)
sweep.to_csv(CFG.artifact_dir / "threshold_sweep_val.csv", index=False)
print(f"Saved operating point → {CFG.ckpt_dir / 'classifier_meta.json'}")
 
 

Loaded best checkpoint from epoch 15
F1-optimal threshold        : 0.050
High-recall (≥0.98) threshold: 0.700

VAL metrics @ tuned threshold:
  loss      : 0.0101
  acc       : 0.9989
  precision : 0.9985
  recall    : 0.9997
  f1        : 0.9991
  auc       : 0.9997

Classification report (val @ tuned threshold):
              precision    recall  f1-score   support

   No-Damage     0.9996    0.9978    0.9987      2267
      Damage     0.9985    0.9997    0.9991      3269

    accuracy                         0.9989      5536
   macro avg     0.9990    0.9987    0.9989      5536
weighted avg     0.9989    0.9989    0.9989      5536


Saved plots → D:\nti_project\audit_results\classification\classification_diagnostics.png
Saved operating point → D:\nti_project\models\classification\classifier_meta.json


In [17]:
model.eval()
onnx_path = CFG.ckpt_dir / f"classifier_{CFG.arch}.onnx"
dummy = torch.randn(1, 3, CFG.img_size, CFG.img_size, device=DEVICE)
 
torch.onnx.export(
    model.float(), dummy, str(onnx_path),
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17, do_constant_folding=True,
)
print(f"ONNX exported → {onnx_path}")
 
# Optional parity check (requires onnxruntime):
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    ref = model(dummy).detach().cpu().numpy()
    got = sess.run(None, {"input": dummy.cpu().numpy()})[0]
    print(f"ONNX/PyTorch max abs diff: {np.abs(ref - got).max():.2e}")
except ImportError:
    print("onnxruntime not installed — skipping parity check "
          "(pip install onnxruntime).")
 
 

ONNX exported → D:\nti_project\models\classification\classifier_resnet50.onnx
ONNX/PyTorch max abs diff: 4.13e-02


In [16]:
%pip install onnx onnxruntime

   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
    --------------------------------------- 0.3/17.2 MB ? eta -:--:--
   -- ------------------------------------- 1.0/17.2 MB 4.2 MB/s eta 0:00:04
   ---- ----------------------------------- 1.8/17.2 MB 4.0 MB/s eta 0:00:04
   ------ --------------------------------- 2.6/17.2 MB 4.1 MB/s eta 0:00:04
   ------- -------------------------------- 3.4/17.2 MB 4.0 MB/s eta 0:00:04
   --------- ------------------------------ 4.2/17.2 MB 4.0 MB/s eta 0:00:04
   ----------- ---------------------------- 5.0/17.2 MB 3.9 MB/s eta 0:00:04
   -------------- ------------------------- 6.0/17.2 MB 4.0 MB/s eta 0:00:03
   --------------- ------------------------ 6.8/17.2 MB 4.0 MB/s eta 0:00:03
   ----------------- ---------------------- 7.6/17.2 MB 3.9 MB/s eta 0:00:03
   ------------------- -------------------- 8.4/17.2 MB 3.9 MB/s eta 0:00:03
   --------------------- ------------------ 9.2/17.2 MB 3.9 MB/s eta 0:00:03
   ----------

In [19]:
from typing import Optional
from pathlib import Path
from PIL import Image
import torch

# 1. تعيين وتثبيت الخصائص المتغيرة لتفادي الـ AttributeError
CFG.seed = getattr(CFG, "seed", 42)
BEST_TH = globals().get("BEST_TH", 0.5)
eval_tf = globals().get("eval_tf", globals().get("val_tfms"))

# 2. المرونة في تحديد الـ DataFrame المتوفر للـ Validation وأعمدته
df_v = val_df if 'val_df' in globals() else df_val
path_col = "abs_path" if "abs_path" in df_v.columns else ("image_path" if "image_path" in df_v.columns else df_v.columns[0])
label_col = "label" if "label" in df_v.columns else ("target" if "target" in df_v.columns else df_v.columns[1])


def predict_image(path: str, threshold: Optional[float] = None) -> dict:
    """Stage-1 gate used by the end-to-end pipeline: run detection/segmentation
    only when this returns is_damaged=True."""
    th = threshold if threshold is not None else BEST_TH
    with Image.open(path) as im:
        x = eval_tf(im.convert("RGB")).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=(getattr(CFG, "amp", True) and DEVICE.type == "cuda")):
        prob = torch.softmax(model(x).float(), dim=1)[0, 1].item()

    is_damaged = bool(prob >= th)
    class_name = CFG.class_names[int(is_damaged)] if hasattr(CFG, "class_names") else ("Damage" if is_damaged else "No-Damage")

    return {
        "damage_probability": round(prob, 4),
        "is_damaged": is_damaged,
        "threshold": th,
        "label": class_name,
    }


# 3. تشغيل الاختبار على عينة عشوائية
sample = df_v.sample(min(5, len(df_v)), random_state=CFG.seed)
for _, r in sample.iterrows():
    img_p = r[path_col]
    out = predict_image(img_p)
    
    raw_target = r[label_col]
    true_label = raw_target if isinstance(raw_target, str) else CFG.class_names[int(raw_target)]
    
    print(f"{Path(img_p).name:45s} true={true_label:10s} "
          f"pred={out['label']:10s} p={out['damage_probability']:.4f}")

print("\n✅ Notebook 04 complete. Test set untouched — evaluate it in Notebook 07.")

M (324).jpg                                   true=Damage     pred=Damage     p=1.0000
C (375).jpg                                   true=Damage     pred=Damage     p=1.0000
Crack_2028.jpg                                true=Damage     pred=Damage     p=1.0000
C (905).jpg                                   true=Damage     pred=Damage     p=1.0000
M (411).jpg                                   true=Damage     pred=Damage     p=1.0000

✅ Notebook 04 complete. Test set untouched — evaluate it in Notebook 07.
